In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
from pathlib import Path
from datetime import timedelta

# ==========================================
# 1. PATH CONFIGURATION
# ==========================================
REQUIRED_FOLDERS = ['data', 'code', 'models', 'images', 'output']
def get_project_root():
    current_path = Path.cwd()
    if current_path.name in REQUIRED_FOLDERS:
        return current_path.parent
    return current_path

BASE_PATH = get_project_root()
DATA_PATH = BASE_PATH / 'data'

# ==========================================
# 2. LOAD DATA
# ==========================================
print("⏳ Loading datasets...")
df_market = pd.read_csv(DATA_PATH / 'market_data_processed.csv')
df_news = pd.read_csv(DATA_PATH / 'news_raw_finviz.csv')

# 2.1 Fix Market Timestamps
# Convert string timestamps back to datetime objects
df_market['timestamp'] = pd.to_datetime(df_market['timestamp'], utc=True).dt.tz_convert('US/Eastern')

# 2.2 Fix News Timestamps
# Challenge: News dates are "Jan-17-26". We need to parse this carefully.
# We'll create a helper function to parse the specific FinViz format.
def parse_finviz_date(date_str, time_str):
    # Combine "Jan-17-26" and "05:35PM" -> "Jan-17-26 05:35PM"
    full_str = f"{date_str} {time_str}"
    try:
        # Parse format: %b-%d-%y %I:%M%p (e.g., Jan-17-26 05:35PM)
        dt_obj = pd.to_datetime(full_str, format='%b-%d-%y %I:%M%p')
    except:
        # Fallback if format varies
        dt_obj = pd.to_datetime(full_str)
    return dt_obj

df_news['datetime'] = df_news.apply(lambda x: parse_finviz_date(x['date'], x['time']), axis=1)

# Localize news to US/Eastern (Finviz is usually EST)
df_news['datetime'] = df_news['datetime'].dt.tz_localize('US/Eastern')

print(f"Data Loaded. Market rows: {len(df_market)}, News rows: {len(df_news)}")

# ==========================================
# 3. BERT SENTIMENT ANALYSIS (NLP)
# ==========================================
print("\nInitializing FinBERT (this may take a moment)...")

# We use ProsusAI/finbert, the industry standard for financial sentiment
model_name = "ProsusAI/finbert"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Create a pipeline for easy usage
nlp_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, return_all_scores=True)

print("Calculating Sentiment Scores for Headlines...")

# Function to get scores
def get_sentiment_scores(text):
    # Truncate text to 512 tokens to fit BERT limit
    results = nlp_pipeline(text[:512])[0]
    # Extract scores: neutral, positive, negative
    scores = {res['label']: res['score'] for res in results}
    return scores['positive'], scores['negative'], scores['neutral']

# Apply to dataframe (This might take 30-60 seconds for 100 rows)
# We use a simple loop or apply.
scores_list = df_news['headline'].apply(get_sentiment_scores)

# Expand the tuple into columns
df_news[['sentiment_pos', 'sentiment_neg', 'sentiment_neu']] = pd.DataFrame(scores_list.tolist(), index=df_news.index)

print("Sentiment Analysis Complete.")
print(df_news[['headline', 'sentiment_pos', 'sentiment_neg']].head(3))

# ==========================================
# 4. ALIGNMENT (NEWS -> MARKET)
# ==========================================
print("\nAligning News to Market Time (5-min intervals)...")

# Sort both by time
df_market = df_market.sort_values('timestamp')
df_news = df_news.sort_values('datetime')

# We want to aggregate news features into the market 5-min buckets.
# If multiple headlines happen in the same 5 mins, we take the mean sentiment.
# If no news happens, the sentiment is 0 (or neutral).

# Set indices for resampling
df_news_indexed = df_news.set_index('datetime')

# Resample news to 5min, taking the mean of sentiment scores
# We use 'label='right', closed='right'' to match market candle close times
df_news_resampled = df_news_indexed[['sentiment_pos', 'sentiment_neg', 'sentiment_neu']].resample('5min', label='right', closed='right').mean()

# Reset index to merge
df_news_resampled.reset_index(inplace=True)
df_news_resampled.rename(columns={'datetime': 'timestamp'}, inplace=True)

# Merge with Market Data
# We use "left" join: we keep all market rows, and attach news if available.
df_final = pd.merge(df_market, df_news_resampled, on='timestamp', how='left')

# Fill NaNs
# If no news occurred in that 5 mins, what is the sentiment?
# Logic: Absence of news = Neutral state.
# However, FinBERT "neutral" score is usually high.
# Let's fill NaNs with 0 for Pos/Neg, and 1 for Neutral (or just 0s if we treat it as a signal magnitude).
# Better approach for ML: Fill with 0. This implies "No Signal".
df_final[['sentiment_pos', 'sentiment_neg', 'sentiment_neu']] = df_final[['sentiment_pos', 'sentiment_neg', 'sentiment_neu']].fillna(0)

# ==========================================
# 5. FINAL EXPORT
# ==========================================
save_path = DATA_PATH / 'final_dataset_ready.csv'
df_final.to_csv(save_path, index=False)

print("\n--- Final Dataset Statistics ---")
print(f"Total Rows: {len(df_final)}")
print(f"Rows with News Signals: {len(df_news_resampled[df_news_resampled['sentiment_pos'] > 0])}")
print(f"Saved to: {save_path}")

C:\Users\Yahya\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Yahya\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Yahya\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please 

⏳ Loading datasets...
✅ Data Loaded. Market rows: 4591, News rows: 100

🤖 Initializing FinBERT (this may take a moment)...


Device set to use cpu
c:\Users\Yahya\anaconda3\envs\myenv\Lib\site-packages\transformers\pipelines\text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


🚀 Calculating Sentiment Scores for Headlines...


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

✅ Sentiment Analysis Complete.
                                            headline  sentiment_pos  \
0  Here's How Much $1,000 in a Trump Account Coul...       0.047988   
1  Investors Worried About Large Cap Concentratio...       0.133787   
2    VOO vs. SPY: What's the Better S&P 500 ETF Buy?       0.056060   

   sentiment_neg  
0       0.017276  
1       0.022496  
2       0.028745  

🔗 Aligning News to Market Time (5-min intervals)...

--- Final Dataset Statistics ---
Total Rows: 4591
Rows with News Signals: 97
💾 Saved to: c:\Users\Yahya\Desktop\My folder\WQU\10. Capstone\Thesis Work\Machine Learning & Deep Learning in Finance\NLP for Intraday Volatility Prediction\data\final_dataset_ready.csv


c:\Users\Yahya\anaconda3\envs\myenv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Yahya\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
